# 05 — Fit / Shortlisting Predictor (Optional Supervised)

Predict whether a candidate is a **good fit** for a specific job using engineered features.

In [ ]:
import sys
sys.path.insert(0,"..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import load_npz
import joblib
import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 1. Prepare Training Data

We create synthetic labels based on cosine similarity threshold.
If match_score >= 0.4 → fit=1, else fit=0.

In [ ]:
from src.config import JOB_CORPUS_CSV, TFIDF_PATH, MODELS_DIR, RESUME_PROC_CSV
from src.features.text_features import load_vectorizer
from sklearn.metrics.pairwise import cosine_similarity

vectorizer = load_vectorizer()
job_matrix = load_npz(str(MODELS_DIR/"job_tfidf_matrix.npz"))
job_corpus  = pd.read_csv(JOB_CORPUS_CSV)
resumes    = pd.read_csv(RESUME_PROC_CSV)

print(f"Resumes: {len(resumes)}, Jobs: {len(job_corpus)}")

In [ ]:
# For each resume, pick a random job and compute features
import random
random.seed(42)

from src.features.text_features import transform_tfidf
from src.features.match_features import skill_overlap, experience_match

rows = []
n_samples = min(2000, len(resumes))
for i, (_, r_row) in enumerate(resumes.sample(n_samples, random_state=42).iterrows()):
    resume_text  = r_row.get("clean_resume", "")
    resume_vec   = transform_tfidf(vectorizer, [resume_text])
    cos_sims     = cosine_similarity(resume_vec, job_matrix).flatten()
    # Positive: top match, Negative: random low match
    pos_idx = cos_sims.argmax()
    neg_idx = cos_sims.argmin()
    for job_idx, label in [(pos_idx, 1), (neg_idx, 0)]:
        j_row = job_corpus.iloc[job_idx]
        job_text = str(j_row.get("description","")) + " " + str(j_row.get("skills",""))
        rows.append({
            "cosine_sim":       float(cos_sims[job_idx]),
            "skill_overlap":    skill_overlap(resume_text, job_text),
            "experience_match": experience_match(resume_text, str(j_row.get("experience",""))),
            "label":            label,
        })

df_fit = pd.DataFrame(rows)
print(df_fit.head())
print("
Label distribution:")
print(df_fit["label"].value_counts())

## 2. Train the Fit Predictor

In [ ]:
from src.models.fit_predictor import train_fit_predictor
X = df_fit[["cosine_sim","skill_overlap","experience_match"]]
y = df_fit["label"]
fit_model, results = train_fit_predictor(X, y)
print("Best model:", results["best_model"])
print("ROC-AUC:   ", results["roc_auc"])

## 3. ROC-AUC Comparison

In [ ]:
models_compared = {k:v["cv_roc_auc_mean"] for k,v in results.items() if isinstance(v,dict) and "cv_roc_auc_mean" in v}
fig,ax = plt.subplots(figsize=(8,4))
ax.bar(models_compared.keys(), models_compared.values(), color=["#6366f1","#a855f7"])
ax.set_ylabel("ROC-AUC")
ax.set_title("Fit Predictor: Model Comparison")
ax.set_ylim(0,1)
plt.tight_layout()
plt.savefig("../reports/figures/fit_predictor_roc.png", dpi=150)
plt.show()

## 4. Predict Fit Score for a Sample

In [ ]:
from src.models.fit_predictor import predict_fit_score
import numpy as np

sample_features = np.array([[0.72, 0.5, 1.0]])  # cosine_sim, skill_overlap, experience_match
fit_score = predict_fit_score(sample_features, fit_model)
print(f"Fit Probability: {fit_score*100:.1f}%")

## ✅ Fit Predictor complete!

Now run the **Streamlit portal**:
